W&B 在工程上不仅仅是一个“画曲线的网站”，它是一套完整的 **MLOps（机器学习运维）基础设施**。它的核心体系由四个部分组成：**Runs（实验追踪）**、**Artifacts（产物与版本控制）**、**Sweeps（超参自动化搜索）** 和 **Reports（动态报告）**。

---

## 一、 W&B 的四大核心支柱

### 1. Runs（基础实验追踪）

这是最常用的部分。每一次你运行 `python train.py`，W&B 就会在云端创建一个独一无二的 **Run ID**。这个 Run 会死死盯住你的进程，实时抓取你的指标、系统硬件指标（DDP 模式下甚至能抓取每张卡的显存和通信状态）、控制台的标准输出（`stdout` / `stderr`）以及报错信息。

### 2. Artifacts（模型与数据集版本控制）

这是工业界极度看重的功能。传统的 `torch.save(model.state_dict(), 'best.pt')` 很容易重名或者搞混。
W&B 的 Artifacts 就像是**针对大数据、大模型的 Git**。它可以把你的数据集、训练好的模型 Checkpoint 自动上传到云端，并打上版本号（如 `v0`, `v1`, `latest`），甚至会**自动生成血统图（Lineage Graph）**，清晰地展示“这个模型是用哪一个数据集、哪一次 Run 训练出来的”。

### 3. Sweeps（自动化超参调优）

你不需要自己写 `for` 循环或者用 shell 脚本去试参数。Sweeps 允许你在云端定义一个搜索空间（比如学习率在 `1e-4` 到 `1e-2` 之间，优化器选 `AdamW` 或 `SGD`），W&B 会自动调度你的多台服务器，采用 **贝叶斯优化（Bayesian Optimization）**、**随机搜索** 或 **网格搜索** 自动帮你寻找最优超参组合，并带有**早停机制（Hyperband）**，发现模型跑崩了会自动掐断，省钱省时。

### 4. Reports（团队协作与技术沉淀）

它可以将你不同时期、不同项目跑出来的动态图表一键拖拽组合在一起。你可以在图表旁边写 Markdown Markdown 注释、插入数学公式，生成一个精美的在线网页。




---

## 一、 什么是 Reports？

在模型训练、调参、Case 分析都完成后，紧接着最痛苦的就是写 PPT 或写论文汇报成果。`wandb.Reports` 就是为了让你彻底告别“手动截图、复制粘贴数据”而生的**现代化动态文档工具**（类似于针对机器学习项目的 **Notion** 或 **石墨文档**）。

### 它的核心优势是：

* **动态数据联动**：它不是静态的截图。如果你的实验（Runs）还在跑，报告里的 Loss 曲线和数据表会**自动实时刷新**。
* **交互式探索**：发给导师或同事的报告链接，他们在浏览器里打开后，依然可以鼠标悬浮看数值、放大缩小曲线，甚至直接在报告里对数据表格进行过滤（Filter）和排序。
* **低代码/免代码组装**：既支持在网页端通过极其直观的拖拽和富文本编辑，也支持在 Python 代码中通过“积木块（Blocks）”对象化地直接生成。

---

## 二、 核心 API 详解与作用（基于最新标准 API）

W&B 的报告在底层采用的是“组件积木块（Blocks）”架构：

### 1. 报告载体：`wr.Report()`

```python
import wandb.apis.reports as wr
report = wr.Report(project="项目名", title="标题", description="描述")

```

* **作用**：在内存中声明一个报告档案，绑定到指定的 W&B 云端项目空间。

### 2. 看板网格：`wr.PanelGrid()`

```python
panel_grid = wr.PanelGrid(runsets=[wr.Runset(project="项目名")], panels=[line_plot])

```

* **作用**：报告中存放图表的“容器网格”。它需要绑定两样东西：数据源（`runsets`，即从哪个项目捞实验数据）和图表类型（`panels`）。

### 3. 图表组件：`wr.LinePlot()` / `wr.WeavePanelSummaryTable()`

```python
line_plot = wr.LinePlot(title="准确率曲线", x="epoch", y=["val_acc"])

```

* **作用**：具体的活图表积木。`LinePlot` 生成动态折线图；`WeavePanelSummaryTable` 则可以直接把你在实验中 log 过的多媒体交互式 `wandb.Table`（如 Case 分析表）直接嵌入到报告中。

### 4. 文本积木：`wr.H1()`, `wr.P()`, `wr.MarkdownBlock()`

* **作用**：对应文档的标题、正文段落和 Markdown 块，负责撰写实验背景和结论。

### 5. 组装与发布：`report.blocks` 与 `report.save()`

```python
report.blocks = [wr.Title("标题"), panel_grid, wr.P("正文")]
report.save()

```

* **作用**：将所有文本、图表、表格积木按顺序放进列表赋给 `report.blocks`，最后调起 `save()` 一键推送到云端生成永久网页链接。

---

## 三、 标准使用流程

W&B Reports 的标准落地流程通常包含以下三步：

```text
  1. 运行实验并打标签 (Tags) ──> 2. 代码/网页端组装积木 (Blocks) ──> 3. 一键分享永久链接与留评协作

```

1. **打标沉淀数据**：在训练或调参脚本中，使用 `wandb.init(tags=["baseline", "v2"])` 给实验打上标签，方便报告后续精准筛选。
2. **拼装动态报告**：在网页端点击 "Create Report" 或在 Python 脚本中用 Blocks 拼装文字与 `PanelGrid`（图表网格）。
3. **协同评审与导出**：把生成的 Report 链接发给团队。大家可以直接在报告的某一时刻的曲线上**留评讨论**（如：“为什么这里突然震荡了？”），评审完毕后可一键导出为标准 PDF 供周报或论文附录使用。

---

## 四、 核心好处提炼

1. **彻底终结“复现地狱”**：传统的 PPT 汇报，别人看到一张好看的静态曲线图，根本不知道你当时是用什么环境、什么代码、哪组参数跑出来的。而 Report 里的每一个图表都可以直接点进去**追溯到产生它的那个具体 Run**，代码、参数、权重一览无余，实现了全链路的数据血缘（Data Lineage）追溯。
2. **极高的高级汇报效率**：一份写好的 Report 模板可以无限复用。下周跑了新的实验，只需要改一下报告的数据筛选条件（Filter），一秒钟就能生成一份全新的动态技术总结。

下面我们就来编写一段**纯内存模拟的、闭环的工业级代码示例**。

这段代码会干两件事：

1. 先模拟运行两组不同的模型实验（ResNet 和 MobileNet），并在内存里模拟它们的准确率收敛过程；
2. 实验一结束，自动调用 Report API，像拼乐高积木一样，把文字、动态对比折线图、甚至是模型产出的 Case 分析表（`wandb.Table`）一并打包，自动在云端撰写出一篇专业的技术汇报文档。

---

In [1]:
# =====================================================================
# 0. 【环境守护补丁】强行保住旧版 pydantic_core 的兼容性
# =====================================================================
import pydantic_core
import json
if not hasattr(pydantic_core, 'from_json'):
    pydantic_core.from_json = lambda s: json.loads(s if isinstance(s, str) else s.decode('utf-8'))

import numpy as np
import time
import wandb
import wandb.apis.reports as wr  # 🚀 导入 W&B 标准报告 API 库

# 定义统一的项目名称
PROJECT_NAME = "learn_wandb"

# =====================================================================
# 🛠️ 步骤 1：模拟两个模型的训练，并 log 数据和 Case 表格
# =====================================================================
def run_mock_experiments():
    print("\n--- 1. 正在启动后台实验：模拟 ResNet 和 MobileNet 的训练 ---")
    
    architectures = ["ResNet-50", "MobileNetV2"]
    
    for arch in architectures:
        run = wandb.init(
            project=PROJECT_NAME, 
            name=f"Run_{arch}", 
            mode="online",
            entity="Jerry-Auto",
            )
        # 记录配置信息
        wandb.config.update({"architecture": arch, "epochs": 10})
        
        # 同时，我们顺便为这个 Run 建立一个极简的 Case 分析表
        case_table = wandb.Table(columns=["Sample_ID", "Status"])
        
        for epoch in range(1, 11):
            time.sleep(0.1)
            # 模拟 ResNet 性能稍好，MobileNet 体积小但稍弱
            base_acc = 0.85 if arch == "ResNet-50" else 0.78
            val_acc = base_acc + (0.1 / epoch) + np.random.normal(0, 0.01)
            val_acc = float(np.clip(val_acc, 0.0, 1.0))
            
            # 实时记录曲线
            wandb.log({"epoch": epoch, "val_acc": val_acc})
            
            # 模拟追加 Case 数据
            if epoch == 10:
                case_table.add_data(f"{arch}_sample_1", "Good")
                case_table.add_data(f"{arch}_sample_2", "Bad" if arch == "MobileNetV2" else "Good")
        
        # 记录表格资产
        wandb.log({"eval_diagnostics_table": case_table})
        run.finish()
    print("✅ 实验全部结束，指标与 Case 表格已成功沉淀上云。")



wandb: ERROR Failed to import wandb_workspaces.  To edit reports programmatically, please install it using `pip install wandb[workspaces]`.


In [2]:
# 1. 先跑实验，把数据喂给云端
run_mock_experiments()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/zhangjinrui/.netrc.



--- 1. 正在启动后台实验：模拟 ResNet 和 MobileNet 的训练 ---


wandb: Currently logged in as: 1763287396 (Jerry-Auto) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/pydantic/main.py:301: UserWarning: Pydantic serializer warnings:
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


epoch,▁▂▃▃▄▅▆▆▇█
val_acc,█▄▃▂▂▁▂▁▁▂
epoch,10
val_acc,0.86588


/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/pydantic/main.py:301: UserWarning: Pydantic serializer warnings:
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


epoch,▁▂▃▃▄▅▆▆▇█
val_acc,█▅▂▁▂▂▁▁▂▁
epoch,10
val_acc,0.79525


✅ 实验全部结束，指标与 Case 表格已成功沉淀上云。


In [31]:
import wandb
from wandb.apis.reports import Report

PROJECT_NAME = "learn_wandb"          # 你的项目名

# 1. 初始化项目（确保有历史数据）
wandb.init(project=PROJECT_NAME, mode="online")

# 2. 创建报告
report = Report(
    project=PROJECT_NAME,
    title="📊 深度学习模型架构评审技术报告",
    description="自动整合收敛曲线，支持在线编辑补充结论。"
)

# 3. 添加一个安全的标准折线图（无需手动声明 runset，自动拉取全部 run）
report.add_panel(
    panel=wandb.plot.line_series(
        xs=[],                         # 留空则自动从所有 run 的 history 中提取 x
        ys=[],                         # 留空则自动抓取 y 对应的 key
        keys=["val_acc"],              # 要绘制的指标 key
        title="📈 验证集准确率收敛曲线",
        xname="epoch"                  # x 轴对应 history 中的字段
    ),
    section="核心指标"                  # 面板归属的章节（可选）
)

# 4. 保存并发布
report.save()
print("✅ 报告创建成功！请前往 W&B 网页端左侧 'Reports' 查看。")
wandb.finish()

ImportError: cannot import name 'Report' from 'wandb.apis.reports' (/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/wandb/apis/reports/__init__.py)

这个报告导出功能调不出来，应该是API已经没了